# Aprendizaje de Simbolo: `ParabolicMLPSymbol` con datos GRF (calor 2D)

Mismo experimento que `02_calor_1d.ipynb` en $\mathbb{T}^2$, con referencia $a(\mathbf{k})=|\mathbf{k}|^2$
(`HeatSymbol`, $n=2$, $m=2$, $\rho=n+1=3$ para el simbolo aprendido).

Reparametrizacion, $\kappa$ y funcion de perdida ($\mathcal{L}_{\text{traj}}$, $\mathcal{L}_{\text{par}}$,
$\mathcal{L}_{\text{class}}$): ver `02_calor_1d.ipynb`, introduccion y Setup.

# Setup
El codigo de cada seccion esta en `parabolic_symbol.learning.experiment.GRFExperiment`.

- **Datos:** GRF en $\mathbb{T}^2=[0,2\pi)^2$, malla de $128\times128$ puntos, $k_{\text{cap}} = 32$.
  Generacion y tiempos: ver `01_exploracion.ipynb`, seccion 2.1. Todos los pares $(k,t)$ con
  $|k|_\infty\le k_{\text{cap}}$, y $k_{\max}=k_{\text{cap}}$.
- **Particion, $\kappa$, $C_\alpha$, barrido, early stopping y seleccion:** como en `02_calor_1d.ipynb`
  (Setup y secciones 2 a 4), con $m=2$.
- `cuda` si esta disponible y `float64`.

In [ ]:
from pathlib import Path

import torch

from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.symbols.fractional_laplacian import FractionalLaplacianSymbol
from parabolic_symbol.learning.experiment import GRFExperiment, summary_table
from parabolic_symbol.utils import default_device

device = default_device()
dtype = torch.float64
print(f"device = {device}  |  dtype = {dtype}")

RESULTS = Path("../results/03_calor_2d")   # tablas (CSV) y figuras (PNG)
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
calor = GRFExperiment(
    reference=HeatSymbol(n=2), m=2.0, resolution=128, k_cap=32,
    results=RESULTS, dtype=dtype, device=device,
)

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [ ]:
display(calor.load_data())

## 2. $\kappa$ teorico y constantes $C_\alpha$

Banda $(c, C)$, $\kappa$ y tabla de $C_\alpha$: ver `02_calor_1d.ipynb`, seccion 2.

In [ ]:
display(calor.estimate_constants())

## 3. Barrido de hiperparametros

Ejes del barrido: ver `02_calor_1d.ipynb`, seccion 3.

In [ ]:
calor.sweep()

## 4. Resultados del barrido

In [ ]:
barrido, por_eje = calor.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Criterio de seleccion: ver `02_calor_1d.ipynb`, seccion 4.

In [ ]:
display(calor.select())

### Metricas del modelo seleccionado

Reportes de trayectoria y de simbolo: ver `02_calor_1d.ipynb`, seccion 4. Aqui el simbolo se compara con $|\mathbf{k}|^2$.

In [ ]:
traj, sym = calor.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Errores en prueba: ver `02_calor_1d.ipynb`, seccion 4. El error en espacio fisico se mide en la malla de $128\times128$.

In [ ]:
display(calor.test())

## 5. Curvas de entrenamiento

Perdida y criterio de early stopping: ver `02_calor_1d.ipynb`, seccion 5.

In [ ]:
calor.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^2$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
calor.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla de certificacion: ver `02_calor_1d.ipynb`, seccion 7.

In [ ]:
display(calor.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

Verificacion de la clase: ver `01_exploracion.ipynb`, seccion 5, y `02_calor_1d.ipynb`, seccion 8.

In [ ]:
display(calor.symbol_class())